## 1. My lane and why

I have selected **Lane 2: Refresh / Content Opportunity Scoring**.

I selected this lane because content teams usually have limited time and cannot review every page at once. My project will explore whether observable search, content freshness, visibility, and engagement signals can help create a ranked list of pages that may deserve human review.

The purpose is not simply to train a machine-learning model. The purpose is to improve the decision about which page should be reviewed first and why. The final output should support an editor with clear scores, suggested actions, and reason codes.

In [8]:
%pip install -r ../../requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import subprocess
import pandas as pd

# Your renamed GitHub repository
REPO_URL = "https://github.com/mimranofficial01-sys/flyrank-ml-internship01.git"
REPO_DIR = Path("/content/flyrank-ml-internship01")
RELATIVE_CSV_PATH = Path("data/raw/content_refresh_anonymized.csv")


def find_file_upwards(start_path, relative_path):
    """Search the current folder and its parent folders for a file."""
    start_path = start_path.resolve()

    for folder in [start_path, *start_path.parents]:
        candidate = folder / relative_path

        if candidate.exists():
            return candidate

    return None


csv_path = find_file_upwards(Path.cwd(), RELATIVE_CSV_PATH)

# In Colab, clone the repository when the dataset is not present.
if csv_path is None:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_DIR)],
            check=True
        )

    csv_path = REPO_DIR / RELATIVE_CSV_PATH

if not csv_path.exists():
    raise FileNotFoundError(
        f"Dataset was not found at: {csv_path}"
    )

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Dataset location:", csv_path)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

Dataset loaded successfully.
Dataset location: C:\Users\lenovo\flyrank-ml-internship01\data\raw\content_refresh_anonymized.csv
Rows: 30,000
Columns: 44


## 2. The question: decision, action, cost of a wrong call

### Research question

Which pseudonymized content pages should an editor review first for refresh, expansion, protection, pruning, or monitoring, based on observable search, freshness, visibility, and engagement signals?

### Unit of analysis

The unit of analysis is one pseudonymized content item or page.

### Decision

The decision is which pages should receive limited editorial attention first.

### Output

The intended output is a ranked review queue containing:

- a priority score;
- a suggested review action;
- understandable reason codes;
- and a confidence level.

### Person and action

A content editor or search specialist would examine the highest-ranked pages, review the supporting evidence, and then decide whether to refresh, expand, protect, prune, consolidate, or monitor each page.

### Cost of a wrong recommendation

A false positive could waste editorial time on a page that does not require attention. A false negative could cause the team to miss a page that is losing meaningful visibility or engagement. An overconfident recommendation could also encourage unnecessary changes to a healthy, seasonal, or low-volume page.

### Why data or ML may help

A simple rule will be used as the first baseline. Machine learning will only be useful if several signals interact in ways that a transparent rule cannot capture well and if the learned ranking performs better under honest validation. Human review will remain part of the final decision.

In [10]:
required_columns = {
    "content_id",
    "client_id",
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "avg_position",
    "ctr"
}

missing_columns = sorted(required_columns.difference(df.columns))

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

print("Unit of analysis: one pseudonymized content item/page")
print("Task type: ranking and opportunity scoring")
print("Primary user: content editor or search specialist")
print("Required columns are available.")

Unit of analysis: one pseudonymized content item/page
Task type: ranking and opportunity scoring
Primary user: content editor or search specialist
Required columns are available.


## 3. Quick look at the data: 2–3 real numbers

The code below examines the starter dataset and calculates several descriptive counts relevant to my selected lane.

I will focus on pages with some search visibility and at least 90 days of content age. I will then count examples such as declining pages with meaningful demand, stale visible pages, and visible pages with relatively low CTR.

These are descriptive indicators showing that a prioritisation problem exists. They are not proof that changing these pages will improve performance.

In [11]:
# Prepare the same basic eligible page population used by the starter workflow.
eligible = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

# Possible review groups
declining_with_demand = (
    eligible["trend_direction"].eq("down") &
    eligible["impressions_90d"].ge(100)
)

stale_visible_pages = (
    eligible["days_since_last_update"].ge(180) &
    eligible["impressions_90d"].ge(500)
)

low_ctr_visible_pages = (
    eligible["impressions_90d"].ge(500) &
    eligible["avg_position"].gt(0) &
    eligible["avg_position"].le(20) &
    eligible["ctr"].lt(0.5)
)


def result_row(name, count, denominator):
    percentage = 100 * count / denominator if denominator else 0

    return {
        "Measure": name,
        "Pages": int(count),
        "Percentage of eligible pages": round(percentage, 2)
    }


summary = pd.DataFrame([
    result_row(
        "All eligible pages",
        len(eligible),
        len(eligible)
    ),
    result_row(
        "Declining pages with at least 100 impressions",
        declining_with_demand.sum(),
        len(eligible)
    ),
    result_row(
        "Pages not updated for 180+ days with 500+ impressions",
        stale_visible_pages.sum(),
        len(eligible)
    ),
    result_row(
        "Visible pages with position 1–20 and CTR below 0.5%",
        low_ctr_visible_pages.sum(),
        len(eligible)
    )
])

display(summary)

,Measure,Pages,Percentage of eligible pages
0,All eligible pages,30000,100.00
1,Declining pages with at least 100 impressions,13152,43.84
2,Pages not updated for 180+ days with 500+ impr...,17,0.06
3,Visible pages with position 1–20 and CTR below...,9759,32.53


The starter dataset contains **[total rows]** rows, of which **[eligible pages]** met the basic eligibility conditions. Among the eligible pages, **[declining count]** were declining while still receiving at least 100 impressions, and **[stale-visible count]** had at least 500 impressions despite not being updated for 180 days or more.

These observed counts suggest that there are enough possible review candidates to justify developing and evaluating a transparent page-prioritisation method over the next seven weeks.

## 4. Careful words: what I can and cannot claim

This project may identify measured patterns and produce a directional, evidence-based ranking of pages for human review. It may show that certain combinations of visibility, freshness, search position, CTR, and engagement signals are associated with greater review priority in the available dataset.

The project will not claim that refreshing a page causes traffic recovery. It will not claim to predict Google's algorithm, guarantee future rankings, or prove that every highly ranked page should be edited.

The final output will be treated as decision support rather than an automatic publishing decision. Minimum-volume filters, client-aware or time-aware validation, reason codes, and human review will be used to reduce misleading recommendations.

In [12]:
quality_checks = pd.DataFrame({
    "Data check": [
        "Rows in original starter dataset",
        "Unique pseudonymized clients",
        "Rows with avg_position = 0 or unavailable",
        "Rows excluded by basic eligibility conditions"
    ],
    "Value": [
        len(df),
        df["client_id"].nunique(),
        int(df["avg_position"].eq(0).sum()),
        int(len(df) - len(eligible))
    ]
})

display(quality_checks)

print("\nInterpretation:")
print("- Position value 0 will not be treated as a real search position.")
print("- Low-volume and very new pages should not automatically receive strong recommendations.")
print("- Results will be described as observational and decision-support only.")

,Data check,Value
0,Rows in original starter dataset,30000
1,Unique pseudonymized clients,32
2,Rows with avg_position = 0 or unavailable,1205
3,Rows excluded by basic eligibility conditions,0



Interpretation:
- Position value 0 will not be treated as a real search position.
- Low-volume and very new pages should not automatically receive strong recommendations.
- Results will be described as observational and decision-support only.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are included
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] The notebook is saved under `work/notebooks/`